# Figure S15

Maps uncertainty and consensus confidence in the groundwater response classes.


In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import pyogrio
from pyproj import Transformer

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS15'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy'
UNCERTAINTY_PATH = RECON / 'model_uncertainty' / 'monthly_model_uncertainty_radius_matrix.npy'
GRID_PATH = RECON / 'metadata' / 'grid_lookup.csv'
MONTH_PATH = RECON / 'metadata' / 'month_index.csv'
BOUNDARY_PATH = ROOT / 'data' / '0 mask MRVA' / 'outerboundary.shp'
PROBABILITY_PATH = (
    RECON / 'metrics' / 'regression'
    / 'best_extratrees_slow_recovery_probabilities.csv'
)
RIVER_PATH = (
    ROOT / 'data_raw' / '11 river_network' / 'HydroRIVERS_NorthAmerica'
    / 'HydroRIVERS_v10_na_shp' / 'HydroRIVERS_v10_na.shp'
)
RIVER_ORDER_FIELD = 'ORD_CLAS'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.bf': 'Arial:bold',
    'axes.unicode_minus': False,
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'xtick.major.size': 3.0,
    'ytick.major.size': 3.0,
    'savefig.dpi': 900,
})


def display_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


print('Reconstruction:', display_path(RECON))
print('Output:', display_path(OUT_DIR))

Reconstruction: outputs\RECON_MAIN_2011_2023
Output: outputs\figures\FigS15


In [2]:
mat = np.load(MATRIX_PATH, mmap_mode='r')
unc = np.load(UNCERTAINTY_PATH, mmap_mode='r')
grid = pd.read_csv(GRID_PATH)
month_index = pd.read_csv(MONTH_PATH)

if mat.shape != unc.shape:
    raise ValueError(
        f'Reconstruction matrix {mat.shape} and uncertainty matrix {unc.shape} do not match.'
    )
if mat.shape[0] != len(month_index):
    raise ValueError(
        f'Matrix months ({mat.shape[0]}) do not match month index ({len(month_index)}).'
    )
if mat.shape[1] != len(grid):
    raise ValueError(
        f'Matrix columns ({mat.shape[1]}) do not match grid rows ({len(grid)}).'
    )

row0, row1 = int(grid['row'].min()), int(grid['row'].max())
col0, col1 = int(grid['col'].min()), int(grid['col'].max())

boundary = gpd.read_file(BOUNDARY_PATH)
if boundary.crs is None:
    boundary = boundary.set_crs('EPSG:5070')
boundary = boundary.to_crs('EPSG:5070')

river_bbox = tuple(float(value) for value in boundary.to_crs('EPSG:4326').total_bounds)
rivers = pyogrio.read_dataframe(
    RIVER_PATH,
    bbox=river_bbox,
    where=f'{RIVER_ORDER_FIELD} = 1',
    columns=[RIVER_ORDER_FIELD],
    use_arrow=True,
)
rivers = rivers.to_crs(boundary.crs)
rivers = gpd.clip(rivers, boundary)
rivers = rivers[rivers.geometry.notna() & (~rivers.geometry.is_empty)].copy()

print(f'Reconstruction shape: {mat.shape}')
print(f'Active MRVA cells: {len(grid):,}')

Reconstruction shape: (156, 87871)
Active MRVA cells: 87,871


In [3]:
LABELS_PATH = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
PI75_NORMAL_FACTOR = 1.150349
PROPAGATION_SEED = 20260711
CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}
METRIC_KEYS = [
    'decline_m', 'Rdown_m_per_month', 'RR_early',
    'T50_months', 'RR2019'
]

month_labels = month_index['month_label'].astype(str)
year_values = month_labels.str[:4].astype(int).to_numpy()
month_values = month_labels.str[5:7].astype(int).to_numpy()
pre_idx = np.flatnonzero(
    (year_values == 2012) & (month_values <= 4)
)
drought_idx = np.flatnonzero(
    (year_values == 2012)
    & (month_values >= 5)
    & (month_values <= 10)
)
recovery_idx = np.flatnonzero(
    month_labels.between('2012-11', '2013-04').to_numpy()
)
baseline_idx = np.flatnonzero(year_values == 2011)
long_deficit_idx = np.flatnonzero(year_values == 2012)
long_recovery_idx = np.flatnonzero(year_values == 2019)
month_numbers = np.arange(mat.shape[0], dtype=int)

def calculate_response_metrics(wtd):
    pre_values = np.asarray(wtd[pre_idx], dtype=np.float32)
    drought_values = np.asarray(wtd[drought_idx], dtype=np.float32)
    recovery_values = np.asarray(wtd[recovery_idx], dtype=np.float32)
    pre_wtd = np.nanmin(pre_values, axis=0)
    pre_peak_idx = pre_idx[np.nanargmin(pre_values, axis=0)]
    drought_peak = np.nanmax(drought_values, axis=0)
    drought_peak_idx = drought_idx[np.nanargmax(drought_values, axis=0)]
    decline = drought_peak - pre_wtd
    valid_decline = decline >= 0.30
    months_to_peak = np.maximum(drought_peak_idx - pre_peak_idx, 1)
    decline_rate = decline / months_to_peak
    recovery_wtd = np.nanmin(recovery_values, axis=0)
    recovery_amount = drought_peak - recovery_wtd
    with np.errstate(divide='ignore', invalid='ignore'):
        early_fraction = recovery_amount / decline
    threshold = drought_peak - 0.5 * decline
    hit = (
        (month_numbers[:, None] > drought_peak_idx[None, :])
        & (wtd <= threshold[None, :])
    )
    first_hit = hit.argmax(axis=0)
    t50 = (first_hit - drought_peak_idx).astype(float)
    t50[~hit.any(axis=0)] = np.nan
    baseline = np.nanmean(
        np.asarray(wtd[baseline_idx], dtype=np.float32), axis=0
    )
    deficit_peak = np.nanmax(
        np.asarray(wtd[long_deficit_idx], dtype=np.float32), axis=0
    )
    deficit = deficit_peak - baseline
    residual_2019 = (
        np.nanmean(
            np.asarray(wtd[long_recovery_idx], dtype=np.float32), axis=0
        ) - baseline
    )
    valid_deficit = deficit >= 0.10
    with np.errstate(divide='ignore', invalid='ignore'):
        recovery_2019 = (deficit - residual_2019) / deficit
    for values in [decline, decline_rate, early_fraction, t50]:
        values[~valid_decline] = np.nan
    recovery_2019[~valid_deficit] = np.nan
    return {
        'decline_m': decline,
        'Rdown_m_per_month': decline_rate,
        'RR_early': early_fraction,
        'T50_months': t50,
        'RR2019': recovery_2019,
    }

def assign_response_classes(metric_values):
    required = [
        'decline_m', 'Rdown_m_per_month',
        'RR_early', 'T50_months'
    ]
    valid = np.isfinite(
        np.column_stack([metric_values[key] for key in required])
    ).all(axis=1)
    rank_values = {
        'decline_m': metric_values['decline_m'],
        'Rdown_m_per_month': metric_values['Rdown_m_per_month'],
        'RR_early': np.clip(metric_values['RR_early'], 0.0, 2.0),
        'T50_months': np.clip(metric_values['T50_months'], 1.0, 72.0),
        'RR2019': np.clip(metric_values['RR2019'], -1.0, 5.0),
    }
    ranks = {
        key: pd.Series(rank_values[key]).rank(
            pct=True, method='average'
        ).to_numpy(dtype=float)
        for key in METRIC_KEYS
    }
    ranks['RR2019_deficit'] = 1.0 - ranks['RR2019']
    ranks['RR2019_deficit'][
        ~np.isfinite(ranks['RR2019_deficit'])
    ] = 0.5
    rank_names = [
        'decline_m', 'Rdown_m_per_month',
        'RR_early', 'T50_months', 'RR2019_deficit'
    ]
    target = np.array([
        [0.78, 0.65, 0.88, 0.12, 0.35],
        [0.62, 0.55, 0.25, 0.88, 0.80],
        [0.10, 0.12, 0.60, 0.28, 0.12],
    ], dtype=float)
    weights = np.array([
        [0.20, 0.16, 0.32, 0.24, 0.08],
        [0.15, 0.10, 0.25, 0.35, 0.15],
        [0.35, 0.30, 0.08, 0.07, 0.20],
    ], dtype=float)
    weights /= weights.sum(axis=1, keepdims=True)
    x = np.column_stack([ranks[key] for key in rank_names])
    distance = np.full(
        (len(x), len(CLASS_ORDER)), np.inf, dtype=float
    )
    distance[valid] = np.sum(
        weights[None, :, :]
        * np.abs(x[valid, None, :] - target[None, :, :]),
        axis=2
    )
    labels = np.full(len(x), -1, dtype=np.int8)
    labels[valid] = np.argmin(
        distance[valid], axis=1
    ).astype(np.int8)
    buffered_gate = (
        valid
        & (ranks['decline_m'] <= 0.22)
        & (ranks['Rdown_m_per_month'] <= 0.28)
        & (ranks['RR2019_deficit'] <= 0.40)
    )
    labels[buffered_gate] = CLASS_ORDER.index('Buffered')
    return labels

base_wtd = np.asarray(mat, dtype=np.float32)
radius = np.asarray(unc, dtype=np.float32)
sigma = np.nan_to_num(
    radius / PI75_NORMAL_FACTOR,
    nan=0.0, posinf=0.0, neginf=0.0
)

In [4]:
CONSENSUS_PATH = OUT_DIR / 'FigS15g_consensus_classification_confidence.png'
LABELS_PATH = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
labels_table = pd.read_csv(LABELS_PATH, usecols=['grid_id', 'response_class'])
class_lookup = labels_table.set_index('grid_id')['response_class']
base_class = grid['grid_id'].map(class_lookup).to_numpy()
base_class_idx = np.array([
    CLASS_ORDER.index(value) if value in CLASS_ORDER else -1
    for value in base_class
], dtype=int)
base_valid = base_class_idx >= 0

ANCHOR_BLOCK_MONTHS = 6
N_ANCHOR_DRAWS = 64
anchor_indices = np.flatnonzero(
    (month_values == 1) | ((year_values == 2023) & (month_values == 12))
).astype(int)
anchor_indices = np.unique(np.r_[anchor_indices, 0, mat.shape[0] - 1])

def anchor_conditioned_noise(rng, n_time, n_cells, anchor_idx, block_months):
    noise = np.zeros((n_time, n_cells), dtype=np.float32)
    for start, stop in zip(anchor_idx[:-1], anchor_idx[1:]):
        control_idx = np.arange(start, stop + 1, block_months, dtype=int)
        if control_idx[-1] != stop:
            control_idx = np.r_[control_idx, stop]
        control = rng.standard_normal(
            (len(control_idx), n_cells)
        ).astype(np.float32)
        control[0] = 0.0
        control[-1] = 0.0
        for control_pos, (left, right) in enumerate(
            zip(control_idx[:-1], control_idx[1:])
        ):
            local_idx = np.arange(left, right + 1, dtype=int)
            fraction = (
                (local_idx - left) / max(float(right - left), 1.0)
            )[:, None]
            noise[left:right + 1] = (
                (1.0 - fraction) * control[control_pos][None, :]
                + fraction * control[control_pos + 1][None, :]
            ).astype(np.float32)
    noise[anchor_idx] = 0.0
    return noise

conditioned_class_counts = np.zeros(
    (mat.shape[1], len(CLASS_ORDER)), dtype=np.int16
)
conditioned_rng = np.random.default_rng(PROPAGATION_SEED + 1)
for _ in range(N_ANCHOR_DRAWS):
    conditioned_noise = anchor_conditioned_noise(
        conditioned_rng, base_wtd.shape[0], base_wtd.shape[1],
        anchor_indices, ANCHOR_BLOCK_MONTHS
    )
    values = calculate_response_metrics(base_wtd + sigma * conditioned_noise)
    draw_labels = assign_response_classes(values)
    for class_idx in range(len(CLASS_ORDER)):
        conditioned_class_counts[:, class_idx] += (draw_labels == class_idx)

consensus_class_idx = conditioned_class_counts.argmax(axis=1)
consensus_confidence = conditioned_class_counts.max(axis=1) / N_ANCHOR_DRAWS
consensus_valid = conditioned_class_counts.sum(axis=1) > 0
consensus_agreement = np.mean(
    consensus_class_idx[base_valid] == base_class_idx[base_valid]
)

n_rows = row1 - row0 + 1
n_cols = col1 - col0 + 1
active_row = (grid['row'] - row0).to_numpy(dtype=int)
active_col = (grid['col'] - col0).to_numpy(dtype=int)
consensus_class_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
consensus_confidence_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
consensus_class_grid[active_row, active_col] = np.where(
    consensus_valid, consensus_class_idx, np.nan
)
consensus_confidence_grid[active_row, active_col] = np.where(
    consensus_valid, consensus_confidence, np.nan
)

consensus_cmap = ListedColormap([
    CLASS_COLORS['Fast recovery'],
    CLASS_COLORS['Slow recovery'],
    CLASS_COLORS['Buffered'],
])
consensus_cmap.set_bad('#FFFFFF')
consensus_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)
confidence_cmap = LinearSegmentedColormap.from_list(
    'consensus_confidence', ['#edf2f3', '#9fc6cc', '#2e6f73']
)
confidence_cmap.set_bad('#FFFFFF')
probability_cmap = LinearSegmentedColormap.from_list(
    'prob_slow_recovery', ['#F8F2E8', '#E7A3B8', '#9F2F5E']
)
probability_cmap.set_bad('#FFFFFF')


x_by_col = grid.groupby('col')['x'].first().sort_index().to_numpy(dtype=float)
y_by_row = grid.groupby('row')['y'].first().sort_index().to_numpy(dtype=float)
dx = float(np.nanmedian(np.diff(x_by_col)))
dy = float(np.nanmedian(np.diff(y_by_row)))
x_edges = np.r_[x_by_col[0] - 0.5 * dx, x_by_col + 0.5 * dx]
y_edges = np.r_[y_by_row[0] - 0.5 * dy, y_by_row + 0.5 * dy]
xx_edges, yy_edges = np.meshgrid(x_edges, y_edges)
transformer = Transformer.from_crs(
    'EPSG:5070', 'EPSG:4326', always_xy=True
)
lon_edges, lat_edges = transformer.transform(xx_edges, yy_edges)
lon_edges = np.asarray(lon_edges)
lat_edges = np.asarray(lat_edges)
map_extent_lonlat = (
    float(np.nanmin(lon_edges)),
    float(np.nanmax(lon_edges)),
    float(np.nanmin(lat_edges)),
    float(np.nanmax(lat_edges)),
)
mean_lat = 0.5 * (map_extent_lonlat[2] + map_extent_lonlat[3])
boundary_lonlat = boundary.to_crs('EPSG:4326')
rivers_lonlat = rivers.to_crs('EPSG:4326')

original_class_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
original_class_grid[active_row, active_col] = np.where(
    base_valid, base_class_idx, np.nan
)
probability_table = pd.read_csv(
    PROBABILITY_PATH,
    usecols=[
        'grid_id',
        'P_slow_recovery_extratrees_resistivity_pumping',
    ],
)
if not probability_table['grid_id'].is_unique:
    raise ValueError('Slow-recovery probability grid_id values are not unique.')
probability_lookup = probability_table.set_index('grid_id')[
    'P_slow_recovery_extratrees_resistivity_pumping'
]
slow_probability = grid['grid_id'].map(probability_lookup).to_numpy(dtype=float)
slow_probability_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
slow_probability_grid[active_row, active_col] = np.where(
    base_valid, slow_probability, np.nan
)

def plot_geographic_context(axis):
    if not rivers_lonlat.empty:
        rivers_lonlat.plot(
            ax=axis, color='#72B9D3', linewidth=0.68,
            alpha=0.98, zorder=3
        )
    boundary_lonlat.boundary.plot(
        ax=axis, color='#1f1f1f', linewidth=0.45, zorder=4
    )
    axis.set_xlim(map_extent_lonlat[0], map_extent_lonlat[1])
    axis.set_ylim(map_extent_lonlat[2], map_extent_lonlat[3])
    axis.set_xticks([])
    axis.set_yticks([])
    axis.tick_params(
        axis='both', which='both', bottom=False, left=False,
        labelbottom=False, labelleft=False
    )
    axis.set_aspect(1.0 / np.cos(np.deg2rad(mean_lat)))
    for spine in axis.spines.values():
        spine.set_visible(False)

fig_consensus, (
    ax_original, ax_consensus, ax_confidence, ax_probability
) = plt.subplots(1, 4, figsize=(12.0, 5.0))
fig_consensus.subplots_adjust(
    left=0.015, right=0.985, bottom=0.035, top=0.94, wspace=0.22
)

ax_original.pcolormesh(
    lon_edges, lat_edges, original_class_grid,
    cmap=consensus_cmap, norm=consensus_norm,
    shading='flat', rasterized=True
)
ax_consensus.pcolormesh(
    lon_edges, lat_edges, consensus_class_grid,
    cmap=consensus_cmap, norm=consensus_norm,
    shading='flat', rasterized=True
)
im_confidence = ax_confidence.pcolormesh(
    lon_edges, lat_edges, consensus_confidence_grid,
    cmap=confidence_cmap, vmin=1.0 / N_ANCHOR_DRAWS,
    vmax=1.0, shading='flat', rasterized=True
)
im_probability = ax_probability.pcolormesh(
    lon_edges, lat_edges, slow_probability_grid,
    cmap=probability_cmap, vmin=0.0, vmax=1.0,
    shading='flat', rasterized=True
)

for axis in [ax_original, ax_consensus, ax_confidence, ax_probability]:
    plot_geographic_context(axis)

ax_original.set_title(
    'Original response classes',
    loc='left', fontweight='bold', fontsize=9, pad=4
)
ax_consensus.set_title(
    'Perturbed consensus classes',
    loc='left', fontweight='bold', fontsize=9, pad=4
)
ax_confidence.set_title(
    'Classification confidence',
    loc='left', fontweight='bold', fontsize=9, pad=4
)
ax_probability.set_title(
    'Slow-recovery probability',
    loc='left', fontweight='bold', fontsize=9, pad=4
)

class_handles = [
    Patch(facecolor=CLASS_COLORS[name], edgecolor='none', label=name)
    for name in CLASS_ORDER
]
ax_original.legend(
    handles=class_handles, frameon=False, loc='lower right',
    bbox_to_anchor=(0.98, 0.02), borderaxespad=0.0,
    fontsize=7.5, handlelength=1.0, handletextpad=0.4
)

confidence_cax = ax_confidence.inset_axes([1.05, 0.25, 0.06, 0.50])
cbar_consensus = fig_consensus.colorbar(
    im_confidence, cax=confidence_cax,
    ticks=[0.25, 0.5, 0.75, 1.0], extend='both'
)
cbar_consensus.set_label('Maximum class vote', fontsize=8.5)
cbar_consensus.ax.tick_params(labelsize=7.5)
probability_cax = ax_probability.inset_axes([1.05, 0.25, 0.06, 0.50])
cbar_probability = fig_consensus.colorbar(
    im_probability, cax=probability_cax,
    ticks=np.linspace(0.0, 1.0, 6), extend='both'
)
cbar_probability.set_label(r'$P_{\mathrm{S}}$', fontsize=10)
cbar_probability.ax.tick_params(labelsize=7.5)

fig_consensus.savefig(CONSENSUS_PATH, bbox_inches='tight', dpi=900)
plt.close(fig_consensus)

print('Consensus classification diagnostic saved:')
print('  ' + display_path(CONSENSUS_PATH))
print(f'  Agreement with original hard labels: {consensus_agreement:.3f}')
print(f'  Mean consensus confidence: {np.nanmean(consensus_confidence[consensus_valid]):.3f}')
for class_idx, class_name in enumerate(CLASS_ORDER):
    mask = consensus_valid & (consensus_class_idx == class_idx)
    print(
        f'  {class_name}: fraction={np.mean(mask[base_valid]):.3f}, '
        f'mean confidence={np.nanmean(consensus_confidence[mask]):.3f}'
    )

Consensus classification diagnostic saved:
  outputs\figures\FigS15\FigS15g_consensus_classification_confidence.png
  Agreement with original hard labels: 0.824
  Mean consensus confidence: 0.655
  Fast recovery: fraction=0.428, mean confidence=0.690
  Slow recovery: fraction=0.289, mean confidence=0.642
  Buffered: fraction=0.283, mean confidence=0.615
